In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns
from scipy.stats import ttest_ind

lat_col = "network_latency"
bw_col = "network_bandwidth"
time_col = "time"
alg_col = "algorithm"
len_col = "prompt_length"

def get_experiment_df(voltage, voltage_improv):
    df_voltage = pd.read_json("../results/" + voltage)
    df_voltage[alg_col] = "voltage"


    df_voltage_improv = pd.read_json("../results/" + voltage_improv)
    df_voltage_improv[alg_col] = "voltage_improv"

    df = pd.concat([df_voltage, df_voltage_improv], ignore_index=True)

    df[bw_col] = df[bw_col].apply(lambda x: x * 1e-6)  # convert to Mbps
    df[lat_col] = df[lat_col].apply(lambda x: x * 1e3)  # convert to ms

    # Clean data that has warmup effects
    # Find the mean std for each lat_col, bw_col and exclude all entries that are outside 2 stds
    def clean_warmup_effects(df):
        clean_dfs = []
        for (lat, bw, alg, length), group in df.groupby([lat_col, bw_col, alg_col, len_col]):
            time_mean = group[time_col].mean()
            time_std = group[time_col].std()
            # print(f"For lat: {lat}, bw: {bw}, alg: {alg}, length: {length} => mean: {time_mean}, std: {time_std}")
            clean_group = group[(group[time_col] >= time_mean - 2 * time_std) & (group[time_col] <= time_mean + 2 * time_std)]
            clean_dfs.append(clean_group)
        return pd.concat(clean_dfs, ignore_index=True)

    df = clean_warmup_effects(df)
    return df

In [3]:
def plot_experiment_df(df):
    # Use seaborn barplot so we get mean + confidence intervals (default ci=95)
    for text_len in df[len_col].unique():
        df_len = df[df[len_col] == text_len]

        # Helpers for ordering
        def _to_num(x):
            try:
                return float(str(x).strip().lower().replace("ms", "").replace("s", ""))
            except:
                return x

        lat_order = sorted(df_len[lat_col].unique(), key=_to_num)
        alg_order = list(df_len[alg_col].dropna().unique())
        bw_vals = sorted(df_len[bw_col].unique(), key=_to_num)

        # determine y-limit from raw data (safer for CI visualization)
        ymax = df_len[time_col].max() * 1.10

        fig, axes = plt.subplots(1, len(bw_vals), figsize=(3 * len(bw_vals), 4), sharey=False)
        if len(bw_vals) == 1:
            axes = [axes]

        for ax, bw in zip(axes, bw_vals):
            sub = df_len[df_len[bw_col] == bw]

            sns.barplot(
                data=sub,
                x=lat_col,
                y=time_col,
                hue=alg_col,
                order=lat_order,
                hue_order=alg_order,
                ax=ax,
                errorbar=("ci", 95),
                capsize=0.06,
                palette="tab10",
                err_kws={"color": "k", "linewidth": 1},
            )

            ax.set_title(f"{bw} Mbps")
            ax.set_xlabel(lat_col + " (ms)")
            # ax.set_ylim(0, ymax)
            ax.grid(axis="y", alpha=0.3)
            ax.tick_params(axis="x", rotation=0)

        axes[0].set_ylabel(f"Mean {time_col} ± 95% CI")
        fig.suptitle(f"Text Length: {text_len}", fontsize=12)
        # single legend for the figure
        handles, labels = axes[-1].get_legend_handles_labels()
        fig.legend(handles, labels, title=alg_col, loc="upper right", ncol=len(alg_order))
        # remove per-axis legends
        for a in axes:
            a.legend_.remove()
        plt.tight_layout(rect=[0, 0, 1, 0.92])
        # Save as pdf
        plt.savefig(f"../results/voltage_analysis_text_length_{text_len}.pdf")
        plt.show()

In [16]:
def show_speedup_analysis(df):
    results = []

    for text_len in df[len_col].unique():
        df_len = df[df[len_col] == text_len]
        alg_order=sorted(df_len[alg_col].unique())
        for bw in df_len[bw_col].unique():
            for lat in df_len[lat_col].unique():
                sub = df_len[(df_len[lat_col] == lat) & (df_len[bw_col] == bw)]
                if len(sub[alg_col].unique()) < 2:
                    continue
                vals0 = sub[sub[alg_col] == alg_order[0]][time_col]
                vals1 = sub[sub[alg_col] == alg_order[1]][time_col]
                if len(vals0) > 1 and len(vals1) > 1:
                    t_stat, p_val = ttest_ind(vals0, vals1, equal_var=False)
                else:
                    p_val = float('nan')
                mean0 = vals0.mean()
                mean1 = vals1.mean()
                if not pd.isna(mean0) and not pd.isna(mean1):
                    speedup = mean0 / mean1 if mean1 != 0 else float('nan')
                    results.append({
                        len_col: int(text_len),
                        bw_col: int(bw),
                        lat_col: int(lat),
                        f"{alg_order[0]}_mean_time": f"{mean0:.3f}",
                        f"{alg_order[1]}_mean_time": f"{mean1:.3f}",
                        "speedup": f"{(speedup-1)*100:.2f}%",
                        "p_value": f"{p_val:.3f}"
                    })

    speedup_df = pd.DataFrame(results)
    # Export as csv
    speedup_df.to_csv("../results/voltage_speedup_analysis.csv", index=False)
    display(speedup_df)

In [17]:

cpu_2_df = get_experiment_df("quest/voltage_cpu_2.json", "quest/voltage_improv_cpu_2.json")

orin_cuda_2 = get_experiment_df("orin_voltage_4.json", "orin_voltage_improv_4.json")
orin_cuda_4 = get_experiment_df("orin_cuda_voltage_4.json", "orin_cuda_voltage_improv_4.json")

orin_sim_4_df = get_experiment_df("voltage_orin_sim_4.json", "voltage_improv_orin_sim_4.json")
orin_4_cpu_df = get_experiment_df("orin_voltage_2_cpu.json", "orin_voltage_improv_2_cpu.json")

quest_4_cuda_small = get_experiment_df("quest/voltage_cuda_4_2.json", "quest/voltage_improv_cuda_4_2.json")
quest_4_cuda_large = get_experiment_df("quest/voltage_cuda_4.json", "quest/voltage_improv_cuda_4.json")

merge_quest_4 = pd.concat([quest_4_cuda_small, quest_4_cuda_large], ignore_index=True)

quest_4_cuda_sim = get_experiment_df("quest/voltage_cuda_4_sim.json", "quest/voltage_improv_cuda_4_sim.json")
quest_4_cuda_sim_2 = get_experiment_df("quest/voltage_cuda_4_sim_2.json", "quest/voltage_improv_cuda_4_sim_2.json")


show_speedup_analysis(cpu_2_df)



,prompt_length,network_bandwidth,network_latency,voltage_mean_time,voltage_improv_mean_time,speedup,p_value
0,269,10,1,1.859,1.796,3.48%,0.258
1,269,10,5,1.892,1.830,3.38%,0.000
2,269,10,20,2.144,2.083,2.93%,0.000
3,269,100,1,1.350,1.296,4.19%,0.000
4,269,100,5,1.415,1.361,3.96%,0.000
5,269,100,20,1.675,1.615,3.71%,0.000
6,269,1000,1,1.305,1.290,1.18%,0.000
7,269,1000,5,1.376,1.318,4.42%,0.000
8,269,1000,20,1.632,1.569,4.03%,0.000
9,490,10,1,3.010,2.915,3.24%,0.000


In [26]:
orin_cuda_100_1_voltage = get_experiment_df("orin_cuda_100_1_voltage.json", "orin_cuda_100_1_voltage_improv.json")
orin_cuda_100_10_voltage = get_experiment_df("orin_cuda_100_10_voltage.json", "orin_cuda_100_10_voltage_improv.json")
orin_cuda_1000_1_voltage = get_experiment_df("orin_cuda_1000_1_voltage.json", "orin_cuda_1000_1_voltage_improv.json")

show_speedup_analysis(orin_cuda_100_1_voltage)
show_speedup_analysis(orin_cuda_100_10_voltage)
show_speedup_analysis(orin_cuda_1000_1_voltage)

,prompt_length,network_bandwidth,network_latency,voltage_mean_time,voltage_improv_mean_time,speedup,p_value
0,125,0,1000,0.633,0.626,1.14%,0.000
1,173,0,1000,0.839,0.833,0.67%,0.027
2,269,0,1000,1.252,1.244,0.66%,0.001
3,490,0,1000,2.139,2.137,0.12%,0.403
4,1002,0,1000,3.852,3.869,-0.44%,0.000


,prompt_length,network_bandwidth,network_latency,voltage_mean_time,voltage_improv_mean_time,speedup,p_value
0,125,0,10000,1.230,1.232,-0.23%,0.388
1,173,0,10000,1.366,1.351,1.12%,0.000
2,269,0,10000,1.704,1.691,0.80%,0.000
3,490,0,10000,2.446,2.433,0.54%,0.000
4,1002,0,10000,4.024,4.011,0.32%,0.178


,prompt_length,network_bandwidth,network_latency,voltage_mean_time,voltage_improv_mean_time,speedup,p_value
0,125,0,1000,0.269,0.268,0.27%,0.421
1,173,0,1000,0.317,0.316,0.15%,0.619
2,269,0,1000,0.426,0.435,-2.09%,0.000
3,490,0,1000,0.642,0.657,-2.33%,0.000
4,1002,0,1000,1.045,1.066,-1.93%,0.000
